# T5 DPO Training Notebook

This notebook continues the supervised T5 baseline v2 checkpoint with Direct Preference Optimization (DPO).

The DPO dataset was built from paired GPT-4o teacher summaries and T5 baseline v2 predictions. For each tweet, the higher-reward summary is used as the chosen response and the lower-reward summary is used as the rejected response.

The goal is to train a T5-DPO checkpoint that can later be evaluated and scored against:
- GPT-4o teacher summaries
- supervised T5 baseline v2 predictions
- the same role-aware reward function

## Code Environment Check

In [1]:
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Current working directory:", os.getcwd())
print("Kaggle working exists:", Path("/kaggle/working").exists())
print("Kaggle input exists:", Path("/kaggle/input").exists())

print("\nGPU check:")
!nvidia-smi

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Current working directory: /kaggle/working
Kaggle working exists: True
Kaggle input exists: True

GPU check:
Tue Jun 30 05:07:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8

## Clone The Project Repository

This cell clones the latest GitHub repository into `/kaggle/working`, enters the project directory, and confirms that the DPO preference dataset and custom DPO training script are available.

In [2]:
from pathlib import Path
import os

REPO_URL = "https://github.com/ffaustin17/role-aware-crisis-summarization.git"
REPO_DIR = Path("/kaggle/working/role-aware-crisis-summarization")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists. Pulling latest changes...")
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}

print("\nLatest commit:")
!git log --oneline --decorate -5

print("\nRepository status:")
!git status --short --branch

print("\nDPO trainer:")
!ls -lh scripts/train_t5_dpo.py

print("\nPreference files:")
!ls -lh data/preferences

print("\nPreference counts:")
!wc -l data/preferences/*.jsonl

Cloning into '/kaggle/working/role-aware-crisis-summarization'...
remote: Enumerating objects: 366, done.
remote: Counting objects: 100% (366/366), done.
remote: Compressing objects: 100% (256/256), done.
remote: Total 366 (delta 180), reused 271 (delta 90), pack-reused 0 (from 0)
Receiving objects: 100% (366/366), 13.89 MiB | 15.90 MiB/s, done.
Resolving deltas: 100% (180/180), done.
/kaggle/working/role-aware-crisis-summarization

Latest commit:
f5fe570 (HEAD -> main, origin/main, origin/HEAD) Add T5 DPO training script
707b578 Add DPO preference dataset builder
d83880e Add future research directions document
cbca2a4 Update timeline with reward scoring reflection
11f722e Add large-scale reward scoring outputs

Repository status:
## main...origin/main

DPO trainer:
-rw-r--r-- 1 root root 18K Jun 30 05:07 scripts/train_t5_dpo.py

Preference files:
total 13M
-rw-r--r-- 1 root root 6.4M Jun 30 05:07 dpo_preferences_t5_v2_vs_gpt4o_reward_v1.jsonl
-rw-r--r-- 1 root root 626K Jun 30 05:07 d

## Locate The Supervised T5 Baseline V2 Checkpoint

DPO training should start from the supervised T5 v2 checkpoint, not from raw `t5-small`.

The checkpoint is expected to be attached as a Kaggle input from the previous T5 training notebook output. This cell searches `/kaggle/input` for directories containing model checkpoint files.

In [3]:
from pathlib import Path

input_root = Path("/kaggle/input")

print("Top-level Kaggle inputs:")
for path in sorted(input_root.iterdir()):
    print(path)

print("\nCandidate model directories containing config.json and model.safetensors:")
candidates = []
for config_path in input_root.rglob("config.json"):
    model_dir = config_path.parent
    has_model = (model_dir / "model.safetensors").exists() or (model_dir / "pytorch_model.bin").exists()
    has_tokenizer = (model_dir / "tokenizer.json").exists() or (model_dir / "spiece.model").exists()
    if has_model and has_tokenizer:
        candidates.append(model_dir)

for i, path in enumerate(candidates, start=1):
    print(f"{i}. {path}")
    for file_name in ["config.json", "generation_config.json", "model.safetensors", "pytorch_model.bin", "tokenizer.json", "spiece.model", "training_args.bin"]:
        file_path = path / file_name
        if file_path.exists():
            print("   ", file_name, f"{file_path.stat().st_size / (1024 * 1024):.2f} MB")

print("\nCandidate count:", len(candidates))

Top-level Kaggle inputs:
/kaggle/input/notebooks

Candidate model directories containing config.json and model.safetensors:
1. /kaggle/input/notebooks/fabricefaustin17/t5-training-and-eval-notebook/t5_small_baseline_v2
    config.json 0.00 MB
    generation_config.json 0.00 MB
    model.safetensors 230.83 MB
    tokenizer.json 2.01 MB
    training_args.bin 0.01 MB
2. /kaggle/input/notebooks/fabricefaustin17/t5-training-and-eval-notebook/t5_small_baseline_v2_smoke
    config.json 0.00 MB
    generation_config.json 0.00 MB
    model.safetensors 230.83 MB
    tokenizer.json 2.01 MB
    training_args.bin 0.01 MB
3. /kaggle/input/notebooks/fabricefaustin17/t5-training-and-eval-notebook/t5_small_baseline_v2/checkpoint-600
    config.json 0.00 MB
    generation_config.json 0.00 MB
    model.safetensors 230.83 MB
    tokenizer.json 2.01 MB
    training_args.bin 0.01 MB
4. /kaggle/input/notebooks/fabricefaustin17/t5-training-and-eval-notebook/t5_small_baseline_v2/checkpoint-900
    config.json 

## Verify Training Dependencies

This cell verifies the core libraries needed by the custom DPO trainer.

The DPO script uses:
- PyTorch
- Hugging Face Transformers
- tqdm

It does not require TRL because the project uses a custom seq2seq DPO training loop for T5.

In [4]:
from pathlib import Path
import torch
import transformers
import tqdm

T5_V2_CHECKPOINT = Path("/kaggle/input/notebooks/fabricefaustin17/t5-training-and-eval-notebook/t5_small_baseline_v2")

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("\nCheckpoint path:", T5_V2_CHECKPOINT)
print("exists:", T5_V2_CHECKPOINT.exists())
print("files:")
!ls -lh {T5_V2_CHECKPOINT}

torch: 2.10.0+cu128
transformers: 5.0.0
cuda available: True
gpu count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Checkpoint path: /kaggle/input/notebooks/fabricefaustin17/t5-training-and-eval-notebook/t5_small_baseline_v2
exists: True
files:
total 233M
drwxr-xr-x 2 nobody nogroup    0 Jun 30 05:07 checkpoint-600
drwxr-xr-x 2 nobody nogroup    0 Jun 30 05:07 checkpoint-900
-rw-r--r-- 1 nobody nogroup 1.6K Jun 30 05:07 config.json
-rw-r--r-- 1 nobody nogroup  151 Jun 30 05:07 generation_config.json
-rw-r--r-- 1 nobody nogroup 231M Jun 30 05:07 model.safetensors
-rw-r--r-- 1 nobody nogroup 2.4K Jun 30 05:07 tokenizer_config.json
-rw-r--r-- 1 nobody nogroup 2.1M Jun 30 05:07 tokenizer.json
-rw-r--r-- 1 nobody nogroup 5.3K Jun 30 05:07 training_args.bin


## DPO Smoke Test

This cell runs a tiny DPO training job using only a small prefix of the preference dataset.

The goal is not model quality. The goal is to verify that:
- the supervised T5 v2 checkpoint loads
- the frozen reference model loads
- chosen/rejected sequence log-probabilities compute correctly
- the custom DPO loss runs
- a checkpoint and training log are saved

In [5]:
%cd /kaggle/working/role-aware-crisis-summarization

!python scripts/train_t5_dpo.py \
  --model-name-or-path /kaggle/input/notebooks/fabricefaustin17/t5-training-and-eval-notebook/t5_small_baseline_v2 \
  --output-dir /kaggle/working/t5_small_dpo_v1_smoke \
  --max-train-samples 32 \
  --max-eval-samples 16 \
  --num-train-epochs 1 \
  --train-batch-size 2 \
  --eval-batch-size 2 \
  --gradient-accumulation-steps 4 \
  --logging-steps 1

/kaggle/working/role-aware-crisis-summarization
Loading weights: 100%|█| 131/131 [00:00<00:00, 265.31it/s, Materializing param=s
Loading weights: 100%|█| 131/131 [00:00<00:00, 1587.16it/s, Materializing param=
Epoch 1: 100%|██████████| 16/16 [00:03<00:00,  4.69it/s, loss=0.6851, acc=0.625]
{
  "epoch": 1,
  "step": 4,
  "eval_loss": 0.6930795535445213,
  "eval_preference_accuracy": 0.4375,
  "eval_policy_margin": 0.010521266609430313,
  "eval_reference_margin": 0.009166296571493149,
  "eval_implicit_reward_margin": 0.00013549851428251714
}
Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  2.12it/s]
Train records: 32
Validation records: 16
Total optimizer steps: 4
Saved DPO checkpoint to: /kaggle/working/t5_small_dpo_v1_smoke


## Inspect Smoke Output

In [6]:
from pathlib import Path

SMOKE_DIR = Path("/kaggle/working/t5_small_dpo_v1_smoke")

print("Smoke dir exists:", SMOKE_DIR.exists())
print("\nFiles:")
!find {SMOKE_DIR} -maxdepth 2 -type f | sort

print("\nTraining log:")
!cat {SMOKE_DIR}/dpo_training_log.jsonl

Smoke dir exists: True

Files:
/kaggle/working/t5_small_dpo_v1_smoke/config.json
/kaggle/working/t5_small_dpo_v1_smoke/dpo_training_log.jsonl
/kaggle/working/t5_small_dpo_v1_smoke/generation_config.json
/kaggle/working/t5_small_dpo_v1_smoke/model.safetensors
/kaggle/working/t5_small_dpo_v1_smoke/tokenizer_config.json
/kaggle/working/t5_small_dpo_v1_smoke/tokenizer.json

Training log:
{"epoch": 1, "step": 1, "learning_rate": 7.500000000000001e-06, "train_loss": 0.6971043348312378, "train_preference_accuracy": 0.375, "train_policy_margin": -0.7289508208632469, "train_reference_margin": -0.6519100479781628, "train_implicit_reward_margin": -0.0077040777541697025}
{"epoch": 1, "step": 2, "learning_rate": 5e-06, "train_loss": 0.6884664744138718, "train_preference_accuracy": 0.75, "train_policy_margin": -0.8391022905707359, "train_reference_margin": -0.9334542453289032, "train_implicit_reward_margin": 0.009435194311663508}
{"epoch": 1, "step": 3, "learning_rate": 2.5e-06, "train_loss": 0.6985

## Full DPO Training: Beta 0.1

This cell runs the first full DPO training pass from the supervised T5 baseline v2 checkpoint.

The run uses:
- beta = 0.1
- one epoch
- custom seq2seq DPO loss
- frozen reference model initialized from the same supervised T5 v2 checkpoint

This is the first T5-DPO checkpoint. After training, it will need to be evaluated and reward-scored as a third system.

In [7]:
%cd /kaggle/working/role-aware-crisis-summarization

!python scripts/train_t5_dpo.py \
  --model-name-or-path /kaggle/input/notebooks/fabricefaustin17/t5-training-and-eval-notebook/t5_small_baseline_v2 \
  --output-dir /kaggle/working/t5_small_dpo_v1_beta_0_1 \
  --beta 0.1 \
  --num-train-epochs 1 \
  --learning-rate 1e-5 \
  --train-batch-size 2 \
  --eval-batch-size 2 \
  --gradient-accumulation-steps 4 \
  --logging-steps 25

/kaggle/working/role-aware-crisis-summarization
Loading weights: 100%|█| 131/131 [00:00<00:00, 1563.64it/s, Materializing param=
Loading weights: 100%|█| 131/131 [00:00<00:00, 1519.62it/s, Materializing param=
Epoch 1: 100%|██████| 1560/1560 [03:40<00:00,  7.09it/s, loss=0.6783, acc=0.685]
{
  "epoch": 1,
  "step": 390,
  "eval_loss": 0.6776786258983128,
  "eval_preference_accuracy": 0.7208121827411168,
  "eval_policy_margin": 0.22689865385820418,
  "eval_reference_margin": -0.09371685028681295,
  "eval_implicit_reward_margin": 0.032061551243709734
}
Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  2.20it/s]
Train records: 3120
Validation records: 394
Total optimizer steps: 390
Saved DPO checkpoint to: /kaggle/working/t5_small_dpo_v1_beta_0_1


## Inspect Full DPO Checkpoint

In [8]:
from pathlib import Path

DPO_DIR = Path("/kaggle/working/t5_small_dpo_v1_beta_0_1")

print("DPO dir exists:", DPO_DIR.exists())

print("\nTop-level files:")
!find {DPO_DIR} -maxdepth 1 -type f | sort

print("\nDisk usage:")
!du -sh {DPO_DIR}

print("\nTraining log tail:")
!tail -n 20 {DPO_DIR}/dpo_training_log.jsonl

DPO dir exists: True

Top-level files:
/kaggle/working/t5_small_dpo_v1_beta_0_1/config.json
/kaggle/working/t5_small_dpo_v1_beta_0_1/dpo_training_log.jsonl
/kaggle/working/t5_small_dpo_v1_beta_0_1/generation_config.json
/kaggle/working/t5_small_dpo_v1_beta_0_1/model.safetensors
/kaggle/working/t5_small_dpo_v1_beta_0_1/tokenizer_config.json
/kaggle/working/t5_small_dpo_v1_beta_0_1/tokenizer.json

Disk usage:
233M	/kaggle/working/t5_small_dpo_v1_beta_0_1

Training log tail:
{"epoch": 1, "step": 25, "learning_rate": 9.630606860158313e-06, "train_loss": 0.6917538928985596, "train_preference_accuracy": 0.545, "train_policy_margin": -0.04801039665937424, "train_reference_margin": -0.07794731691479682, "train_implicit_reward_margin": 0.0029936921736225487}
{"epoch": 1, "step": 50, "learning_rate": 8.970976253298154e-06, "train_loss": 0.690610231757164, "train_preference_accuracy": 0.58, "train_policy_margin": -0.13543406262993812, "train_reference_margin": -0.18861324593424797, "train_implici

## Evaluate T5-DPO On Held-Out Test Set

This cell evaluates the DPO-trained T5 checkpoint on the same held-out test split used for T5 baseline v2.

The generated summaries will be compared against the GPT teacher target summaries using:
- ROUGE
- BLEU
- BERTScore

This tells us whether DPO preserved or changed teacher-target similarity.

## Install Evaluation Dependencies

In [9]:
%cd /kaggle/working/role-aware-crisis-summarization

!pip install -q rouge-score sacrebleu bert-score

/kaggle/working/role-aware-crisis-summarization
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible

## Verify Evaluation Imports

In [10]:
import sacrebleu
import bert_score
import rouge_score
import transformers
import torch

print("sacrebleu:", sacrebleu.__version__)
print("bert_score import: OK")
print("rouge_score import: OK")
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

sacrebleu: 2.6.0
bert_score import: OK
rouge_score import: OK
transformers: 5.0.0
torch: 2.10.0+cu128
cuda available: True


## Evaluate DPO On Test Split

In [11]:
%cd /kaggle/working/role-aware-crisis-summarization

!python scripts/evaluate_t5_baseline.py \
  --data-dir data/modeling/t5_baseline_v2 \
  --model-dir /kaggle/working/t5_small_dpo_v1_beta_0_1 \
  --metrics-csv reports/tables/t5_small_dpo_v1_beta_0_1_metrics.csv \
  --metrics-md reports/tables/t5_small_dpo_v1_beta_0_1_metrics.md \
  --predictions data/modeling/t5_dpo_v1_beta_0_1/test_predictions.jsonl \
  --batch-size 8

/kaggle/working/role-aware-crisis-summarization
Loading weights: 100%|█| 131/131 [00:00<00:00, 1841.49it/s, Materializing param=
config.json: 100%|█████████████████████████████| 482/482 [00:00<00:00, 2.03MB/s]
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 100kB/s]
vocab.json: 899kB [00:00, 11.9MB/s]
merges.txt: 456kB [00:00, 3.29MB/s]
tokenizer.json: 1.36MB [00:00, 6.30MB/s]
model.safetensors: 100%|████████████████████| 1.42G/1.42G [00:05<00:00, 262MB/s]
Loading weights: 100%|█| 389/389 [00:00<00:00, 1480.29it/s, Materializing param=
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
poole

## Inspect DPO Test Metrics

In [12]:
from pathlib import Path

metrics_md = Path("reports/tables/t5_small_dpo_v1_beta_0_1_metrics.md")
predictions = Path("data/modeling/t5_dpo_v1_beta_0_1/test_predictions.jsonl")

print("Metrics MD exists:", metrics_md.exists())
print("Predictions exist:", predictions.exists())

print("\nMetrics:")
print(metrics_md.read_text(encoding="utf-8"))

print("\nPrediction count:")
!wc -l {predictions}

print("\nFirst prediction:")
!head -n 1 {predictions}

Metrics MD exists: True
Predictions exist: True

Metrics:
| metric | value |
|---|---:|
| rouge1_f1 | 0.463752 |
| rouge2_f1 | 0.229735 |
| rougeL_f1 | 0.406215 |
| bleu | 16.312138 |
| bertscore_precision | 0.917875 |
| bertscore_recall | 0.915317 |
| bertscore_f1 | 0.916531 |
| num_examples | 601.000000 |


Prediction count:
601 data/modeling/t5_dpo_v1_beta_0_1/test_predictions.jsonl

First prediction:
{"tweet_id": 3130, "source_row_id": 22897, "role": "Police", "disaster_type": "Collapse", "information_type": "Other Useful Information", "input_text": "Responder Roles: Police\nDisaster Type: Collapse\nTweet: Arrests over Dhaka building collapse: Two owners of garment factories in the building that collapsed in Bangladesh are ...  @tobeymonster\nSecondary Annotations: CPU\nInformation Type: Other Useful Information", "target_text": "Investigate possible criminal activity linked to the building collapse while maintaining scene security in Dhaka.", "prediction_text": "Police should moni

## Generate T5-DPO Predictions For All 6001 Examples

The test metrics show how closely T5-DPO matches the GPT teacher summaries, but DPO may intentionally move away from teacher imitation.

This cell generates T5-DPO predictions for every split:
- train
- validation
- test

The output will be scored with the role-aware reward function and compared against GPT teacher summaries and supervised T5 v2 predictions.

In [13]:
from pathlib import Path
import json
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

repo = Path("/kaggle/working/role-aware-crisis-summarization")
data_dir = repo / "data/modeling/t5_baseline_v2"
model_dir = Path("/kaggle/working/t5_small_dpo_v1_beta_0_1")
output_path = repo / "data/modeling/t5_dpo_v1_beta_0_1/all_predictions.jsonl"

batch_size = 8
max_input_length = 512
max_generation_length = 128
num_beams = 4

def read_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def batched(records, n):
    for i in range(0, len(records), n):
        yield records[i:i+n]

records = []
for split in ["train", "validation", "test"]:
    for record in read_jsonl(data_dir / f"{split}.jsonl"):
        record = dict(record)
        record["split"] = split
        records.append(record)

print("records:", len(records))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(device)
model.eval()

output_path.parent.mkdir(parents=True, exist_ok=True)

written = 0
with output_path.open("w", encoding="utf-8") as out:
    for batch in tqdm(list(batched(records, batch_size)), desc="Generating all DPO predictions"):
        inputs = [record["input_text"] for record in batch]
        encoded = tokenizer(
            inputs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length,
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **encoded,
                max_length=max_generation_length,
                num_beams=num_beams,
            )

        decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        for record, prediction in zip(batch, decoded):
            out_record = {
                "tweet_id": record["tweet_id"],
                "source_row_id": record.get("source_row_id"),
                "split": record["split"],
                "role": record.get("role"),
                "disaster_type": record.get("disaster_type"),
                "information_type": record.get("information_type"),
                "input_text": record["input_text"],
                "target_text": record["target_text"],
                "prediction_text": prediction.strip(),
            }
            out.write(json.dumps(out_record, ensure_ascii=False) + "\n")
            written += 1

print("wrote:", written)
print("output:", output_path)

records: 6001


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Generating all DPO predictions: 100%|██████████| 751/751 [04:43<00:00,  2.65it/s]

wrote: 6001
output: /kaggle/working/role-aware-crisis-summarization/data/modeling/t5_dpo_v1_beta_0_1/all_predictions.jsonl


## Validate All DPO Predictions

In [14]:
from pathlib import Path
import json
from collections import Counter

path = Path("data/modeling/t5_dpo_v1_beta_0_1/all_predictions.jsonl")
records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

print("Prediction rows:", len(records))
print("Unique tweet IDs:", len({record["tweet_id"] for record in records}))
print("Split counts:", dict(Counter(record["split"] for record in records)))
print("Role counts:", dict(Counter(record["role"] for record in records)))
print("Empty predictions:", sum(1 for record in records if not record["prediction_text"].strip()))

print("\nFile size:")
!ls -lh {path}

print("\nFirst prediction:")
!head -n 1 {path}

print("\nLast prediction:")
!tail -n 1 {path}

Prediction rows: 6001
Unique tweet IDs: 6001
Split counts: {'train': 4800, 'validation': 600, 'test': 601}
Role counts: {'Police': 3280, 'Firefighter': 1023, 'Police/Firefighter': 340, 'EMS': 593, 'Firefighter/EMS': 153, 'Police/Firefighter/EMS': 211, 'Police/EMS': 401}
Empty predictions: 0

File size:
-rw-r--r-- 1 root root 4.1M Jun 30 05:18 data/modeling/t5_dpo_v1_beta_0_1/all_predictions.jsonl

First prediction:
{"tweet_id": 5025, "source_row_id": 12926, "split": "train", "role": "Police", "disaster_type": "Floods", "information_type": "Affected individuals", "input_text": "Responder Roles: Police\nDisaster Type: Floods\nTweet: RT @breakingstorm: 2nd death related to Colorado flooding reported - @9NEWS; for more: http://t.co/DfOjwyccDP\nSecondary Annotations: DCC\nInformation Type: Affected individuals", "target_text": "Police should assess public safety threats and coordinate response efforts due to reported fatalities from the flooding.", "prediction_text": "Police should assess s

## Score T5-DPO Predictions With The Role-Aware Reward

This cell scores all 6001 T5-DPO predictions using the same reward configuration used for GPT teacher summaries and supervised T5 v2 predictions:

- tweet-dominant SentenceTransformer relevance
- MiniCheck factuality
- role coverage
- urgency

This is the critical comparison: DPO may reduce traditional target-similarity metrics, but it should ideally improve role-aware reward components.

## Install Reward Scoring Dependencies

In [15]:
%cd /kaggle/working/role-aware-crisis-summarization

!pip install -q sentence-transformers
!pip install -q git+https://github.com/Liyan06/MiniCheck.git

/kaggle/working/role-aware-crisis-summarization
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Verify Reward Imports

In [16]:
import torch
import sentence_transformers

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("sentence_transformers:", sentence_transformers.__version__)

from minicheck.minicheck import MiniCheck
print("MiniCheck import: OK")

torch: 2.10.0+cu128
cuda available: True
sentence_transformers: 5.4.0
MiniCheck import: OK


## Score All T5-DPO Predictions

In [17]:
%cd /kaggle/working/role-aware-crisis-summarization

!python scripts/score_rewards.py \
  --input data/modeling/t5_dpo_v1_beta_0_1/all_predictions.jsonl \
  --output data/rewards/t5_dpo_v1_beta_0_1_reward_scores_tweet_relevance_minicheck.jsonl \
  --summary-csv reports/tables/t5_dpo_v1_beta_0_1_reward_summary_tweet_relevance_minicheck.csv \
  --summary-md reports/tables/t5_dpo_v1_beta_0_1_reward_summary_tweet_relevance_minicheck.md \
  --relevance-backend sentence-transformer \
  --factuality-backend minicheck

/kaggle/working/role-aware-crisis-summarization
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 564kB/s]
README.md: 10.5kB [00:00, 22.5MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 239kB/s]
config.json: 100%|█████████████████████████████| 612/612 [00:00<00:00, 3.75MB/s]
model.safetensors: 100%|███████████████████| 90.9M/90.9M [00:01<00:00, 55.9MB/s]
Loading weights: 100%|█| 103/103 [00:00<00:00, 1441.41it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 2.57MB/s]
vocab.txt: 232kB [00:00, 18.0MB/s]
tokenizer.json: 466kB [00:00, 55.1MB/s]
config.json: 100%|████████████████████

## Inspect T5-DPO Reward Summary

In [18]:
from pathlib import Path
import json
from collections import Counter

path = Path("data/rewards/t5_dpo_v1_beta_0_1_reward_scores_tweet_relevance_minicheck.jsonl")
records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

print("records:", len(records))
print("unique tweet IDs:", len({record["tweet_id"] for record in records}))
print("role counts:", dict(Counter(record["role"] for record in records)))
print("first reward:", records[0]["reward"])
print("first component scores:", records[0]["component_scores"])
print("last tweet_id:", records[-1]["tweet_id"])

print("\nSummary markdown:")
print(Path("reports/tables/t5_dpo_v1_beta_0_1_reward_summary_tweet_relevance_minicheck.md").read_text(encoding="utf-8"))

records: 6001
unique tweet IDs: 6001
role counts: {'Police': 3280, 'Firefighter': 1023, 'Police/Firefighter': 340, 'EMS': 593, 'Firefighter/EMS': 153, 'Police/Firefighter/EMS': 211, 'Police/EMS': 401}
first reward: 0.7643850900232791
first component scores: {'relevance': 0.7016859501600266, 'factuality': 0.4751800298690796, 'role_coverage': 1.0, 'urgency': 1.0}
last tweet_id: 748

Summary markdown:
| group | num_examples | reward_mean | relevance_mean | factuality_mean | role_coverage_mean | urgency_mean |
|---|---:|---:|---:|---:|---:|---:|
| overall | 6001 | 0.653345 | 0.705166 | 0.479749 | 0.751469 | 0.681531 |
| EMS | 593 | 0.668360 | 0.745492 | 0.475642 | 0.675520 | 0.767116 |
| Firefighter | 1023 | 0.668361 | 0.730861 | 0.479511 | 0.546595 | 0.916813 |
| Firefighter/EMS | 153 | 0.543832 | 0.654170 | 0.476482 | 0.459150 | 0.519608 |
| Police | 3280 | 0.677395 | 0.706844 | 0.480405 | 0.902795 | 0.646697 |
| Police/EMS | 401 | 0.584939 | 0.639157 | 0.476725 | 0.683084 | 0.527182 |
|

## Analyze T5-DPO Reward Distribution

This cell creates the same detailed reward analysis tables for the T5-DPO predictions that were created for GPT teacher summaries and supervised T5 v2 predictions.

This allows direct comparison across:
- GPT teacher
- supervised T5 v2
- T5-DPO beta 0.1

In [19]:
%cd /kaggle/working/role-aware-crisis-summarization

!python scripts/analyze_summary_reward_dataset.py \
  --input data/rewards/t5_dpo_v1_beta_0_1_reward_scores_tweet_relevance_minicheck.jsonl \
  --output-dir reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck

print("\nDPO analysis files:")
!find reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck -type f -maxdepth 1 -print | sort

/kaggle/working/role-aware-crisis-summarization
Analyzed records: 6001
Wrote analysis CSVs to: reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck

DPO analysis files:
find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck/by_disaster_type.csv
reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck/by_information_type.csv
reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck/by_role_label.csv
reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck/distributions.csv
reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck/overall_metrics.csv


## GPT vs T5 vs T5-DPO Overall Comparison

In [20]:
import pandas as pd
from pathlib import Path

paths = {
    "gpt4o": Path("reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck/overall_metrics.csv"),
    "t5_v2": Path("reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck/overall_metrics.csv"),
    "t5_dpo_beta_0_1": Path("reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck/overall_metrics.csv"),
}

fields = [
    "reward",
    "relevance",
    "factuality",
    "role_coverage",
    "urgency",
    "summary_word_count",
    "summary_char_length",
    "summary_sentence_count",
]

tables = {
    name: pd.read_csv(path)
    for name, path in paths.items()
}

rows = []
for field in fields:
    row = {"metric": field}
    for name, table in tables.items():
        row[f"{name}_mean"] = float(table.loc[table["numeric_field"] == field, "mean"].iloc[0])
    row["dpo_minus_t5"] = row["t5_dpo_beta_0_1_mean"] - row["t5_v2_mean"]
    row["dpo_minus_gpt"] = row["t5_dpo_beta_0_1_mean"] - row["gpt4o_mean"]
    rows.append(row)

comparison = pd.DataFrame(rows)
display(comparison)

output_path = Path("reports/tables/gpt4o_vs_t5_v2_vs_t5_dpo_beta_0_1_reward_comparison.csv")
comparison.to_csv(output_path, index=False)
print("Wrote:", output_path)

,metric,gpt4o_mean,t5_v2_mean,t5_dpo_beta_0_1_mean,dpo_minus_t5,dpo_minus_gpt
0,reward,0.617690,0.593620,0.653345,0.059726,0.035655
1,relevance,0.724957,0.719472,0.705166,-0.014307,-0.019791
2,factuality,0.478697,0.480305,0.479749,-0.000556,0.001052
3,role_coverage,0.508129,0.471233,0.751469,0.280237,0.243340
4,urgency,0.713276,0.637408,0.681531,0.044123,-0.031745
5,summary_word_count,17.388102,16.719213,17.354441,0.635227,-0.033661
6,summary_char_length,121.824363,117.880187,118.862023,0.981836,-2.962340
7,summary_sentence_count,1.007332,1.011998,1.005666,-0.006332,-0.001666


Wrote: reports/tables/gpt4o_vs_t5_v2_vs_t5_dpo_beta_0_1_reward_comparison.csv


## Three-way role level reward comparison

In [21]:
import pandas as pd
from pathlib import Path

role_paths = {
    "gpt4o": Path("reports/tables/summary_reward_analysis/gpt4o_initial_summaries_v0203_tweet_relevance_minicheck/by_role_label.csv"),
    "t5_v2": Path("reports/tables/summary_reward_analysis/t5_baseline_v2_all_predictions_tweet_relevance_minicheck/by_role_label.csv"),
    "t5_dpo_beta_0_1": Path("reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck/by_role_label.csv"),
}

score_fields = ["reward", "relevance", "factuality", "role_coverage", "urgency"]

merged = None
for name, path in role_paths.items():
    df = pd.read_csv(path)
    df = df[df["numeric_field"].isin(score_fields)][
        ["group_value", "numeric_field", "count", "mean"]
    ].rename(
        columns={
            "group_value": "role",
            "numeric_field": "metric",
            "count": f"{name}_count",
            "mean": f"{name}_mean",
        }
    )
    if merged is None:
        merged = df
    else:
        merged = merged.merge(df, on=["role", "metric"], how="inner")

merged["dpo_minus_t5"] = merged["t5_dpo_beta_0_1_mean"] - merged["t5_v2_mean"]
merged["dpo_minus_gpt"] = merged["t5_dpo_beta_0_1_mean"] - merged["gpt4o_mean"]

output_path = Path("reports/tables/gpt4o_vs_t5_v2_vs_t5_dpo_beta_0_1_reward_comparison_by_role.csv")
merged.to_csv(output_path, index=False)

display(merged)
print("Wrote:", output_path)

,role,metric,gpt4o_count,gpt4o_mean,t5_v2_count,t5_v2_mean,t5_dpo_beta_0_1_count,t5_dpo_beta_0_1_mean,dpo_minus_t5,dpo_minus_gpt
0,EMS,reward,593,0.729448,593,0.639708,593,0.668360,0.028653,-0.061087
1,EMS,relevance,593,0.753576,593,0.757233,593,0.745492,-0.011740,-0.008083
2,EMS,factuality,593,0.476748,593,0.480672,593,0.475642,-0.005030,-0.001105
3,EMS,role_coverage,593,0.909500,593,0.630551,593,0.675520,0.044969,-0.233980
4,EMS,urgency,593,0.823047,593,0.641990,593,0.767116,0.125126,-0.055930
5,Firefighter,reward,1023,0.726688,1023,0.669614,1023,0.668361,-0.001253,-0.058327
6,Firefighter,relevance,1023,0.741738,1023,0.749768,1023,0.730861,-0.018907,-0.010877
7,Firefighter,factuality,1023,0.481340,1023,0.481567,1023,0.479511,-0.002056,-0.001829
8,Firefighter,role_coverage,1023,0.865266,1023,0.592538,1023,0.546595,-0.045943,-0.318671
9,Firefighter,urgency,1023,0.868459,1023,0.841479,1023,0.916813,0.075334,0.048355


Wrote: reports/tables/gpt4o_vs_t5_v2_vs_t5_dpo_beta_0_1_reward_comparison_by_role.csv


## Build T5-DPO Presentation Reward Report

This cell joins T5-DPO predictions with their reward scores into a presentation-ready CSV.

This makes it easier to inspect whether the higher DPO reward reflects genuinely better summaries or reward-seeking behavior such as generic role-keyword phrasing.

In [22]:
%cd /kaggle/working/role-aware-crisis-summarization

!python scripts/build_prediction_reward_report.py \
  --summaries data/modeling/t5_dpo_v1_beta_0_1/all_predictions.jsonl \
  --rewards data/rewards/t5_dpo_v1_beta_0_1_reward_scores_tweet_relevance_minicheck.jsonl \
  --output reports/tables/t5_dpo_v1_beta_0_1_with_rewards_report.csv \
  --expected-count 6001

/kaggle/working/role-aware-crisis-summarization
Summary records: 6001
Reward records: 6001
Wrote report rows: 6001
Wrote report CSV to: reports/tables/t5_dpo_v1_beta_0_1_with_rewards_report.csv


## Preview DPO Report

In [23]:
import pandas as pd
from pathlib import Path

path = Path("reports/tables/t5_dpo_v1_beta_0_1_with_rewards_report.csv")
df = pd.read_csv(path)

print("rows:", len(df))
print("columns:", list(df.columns))
display(df.head(10))

rows: 6001
columns: ['original_tweet', 'input_text', 'candidate_summary', 'reference_summary', 'reward', 'relevance', 'tweet_relevance', 'context_relevance', 'factuality', 'minicheck_support_probability', 'minicheck_predicted_label', 'role_coverage', 'urgency', 'role', 'disaster_type', 'information_type', 'tweet_id', 'source_row_id']


,original_tweet,input_text,candidate_summary,reference_summary,reward,relevance,tweet_relevance,context_relevance,factuality,minicheck_support_probability,minicheck_predicted_label,role_coverage,urgency,role,disaster_type,information_type,tweet_id,source_row_id
0,RT @breakingstorm: 2nd death related to Colora...,Responder Roles: Police\nDisaster Type: Floods...,Police should assess scene security and ensure...,Police should assess public safety threats and...,0.764385,0.701686,0.678118,0.756679,0.475180,0.4751800298690796,0,1.0,1.0,Police,Floods,Affected individuals,5025,12926
1,RT @RichKurtzman: #HighParkFire update: Fire o...,Responder Roles: Firefighter\nDisaster Type: W...,Firefighters should assess fire spread and ens...,Firefighters should evaluate containment strat...,0.687895,0.763486,0.755280,0.782635,0.482699,0.4826991856098175,0,0.5,1.0,Firefighter,Wildfire,Other Useful Information,1963,155
2,RT @ABCNews24: Shane Fitzsimmons: There are mo...,Responder Roles: Firefighter\nDisaster Type: W...,Firefighters should assess fire spread and ens...,Firefighters should evaluate containment effor...,0.692738,0.775822,0.772789,0.782898,0.484801,0.48480144143104553,0,0.5,1.0,Firefighter,Wildfire,Caution and advice,4465,9174
3,New discussion: Police helicopter crash. http:...,"Responder Roles: Police, Firefighter\nDisaster...",Police should monitor for crowd control and en...,Police should assess scene security and coordi...,0.669627,0.709749,0.685402,0.766556,0.484862,0.4848616421222687,0,0.5,1.0,Police/Firefighter,Crash,Other Useful Information,1135,14264
4,"Powerful Quake Hits Costa Rica, Tsunami Warnin...",Responder Roles: Police\nDisaster Type: Earthq...,Police should monitor for crowd control and en...,Police should monitor crowd behavior and poten...,0.690855,0.763774,0.763319,0.764836,0.494136,0.49413609504699707,0,1.0,0.5,Police,Earthquake,Caution and advice,823,1700
5,RT @metrocalgary: UPDATE: Latest from the @cit...,Responder Roles: Police\nDisaster Type: Floods...,Police should assess crowd control and ensure ...,Police should monitor updates for public safet...,0.628412,0.656324,0.595995,0.797092,0.474794,0.47479376196861267,0,1.0,0.4,Police,Floods,Caution and advice,2021,7665
6,#LAXShooting @CNN reporting 3 victims at Ronal...,Responder Roles: EMS\nDisaster Type: Shootings...,Assess casualties and coordinate with EMS to e...,Responders should assess the critical conditio...,0.487213,0.767009,0.769389,0.761455,0.475039,0.47503864765167236,0,0.5,0.0,EMS,Shootings,Affected individuals,1199,15285
7,RT @EcheMadubuike: Explosions have been report...,Responder Roles: Police\nDisaster Type: Bombin...,Police should assess crowd control and ensure ...,Police should assess reported explosions and c...,0.677151,0.720723,0.736218,0.684568,0.499590,0.4995899796485901,0,1.0,0.5,Police,Bombings,Other Useful Information,778,10991
8,RT @NSWRFS: Total fire bans are being declared...,Responder Roles: Police\nDisaster Type: Wildfi...,Police should assess public safety and ensure ...,Police should monitor fire bans and assess pot...,0.807537,0.811239,0.818306,0.794750,0.494413,0.4944133460521698,0,1.0,1.0,Police,Wildfire,Caution and advice,3323,9128
9,RT @shelleylloydabc: Queensland's rainfall out...,Responder Roles: Police\nDisaster Type: Floods...,Police should assess crowd control and ensure ...,Police should monitor rainfall updates to asse...,0.621339,0.637661,0.578198,0.776405,0.472630,0.47263047099113464,0,1.0,0.4,Police,Floods,Other Useful Information,4295,19097


## Build Qualitative Inspection Samples By Role

This cell creates a JSONL file of qualitative inspection examples.

For each exact role label, it selects examples that help diagnose the DPO model:
- strongest DPO improvement over supervised T5
- strongest DPO loss versus supervised T5
- strongest DPO improvement over GPT teacher
- strongest DPO loss versus GPT teacher
- largest DPO relevance drop versus supervised T5
- a median DPO reward example

The output is JSONL rather than a table so full tweet, input, and summary text are preserved without notebook truncation.

In [24]:
import json
from pathlib import Path
from collections import defaultdict

repo = Path("/kaggle/working/role-aware-crisis-summarization")

paths = {
    "gpt_summary": repo / "data/generated/gpt4o_initial_summaries_v0203.jsonl",
    "t5_summary": repo / "data/modeling/t5_baseline_v2/all_predictions.jsonl",
    "dpo_summary": repo / "data/modeling/t5_dpo_v1_beta_0_1/all_predictions.jsonl",
    "gpt_reward": repo / "data/rewards/gpt4o_initial_summaries_v0203_reward_scores_tweet_relevance_minicheck.jsonl",
    "t5_reward": repo / "data/rewards/t5_baseline_v2_all_predictions_reward_scores_tweet_relevance_minicheck.jsonl",
    "dpo_reward": repo / "data/rewards/t5_dpo_v1_beta_0_1_reward_scores_tweet_relevance_minicheck.jsonl",
}

output_path = repo / "reports/tables/t5_dpo_v1_beta_0_1_qualitative_samples_by_role.jsonl"

def read_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def by_tweet_id(path):
    return {int(record["tweet_id"]): record for record in read_jsonl(path)}

def component_scores(record):
    return record.get("component_scores", {})

def candidate_text(summary_record, reward_record):
    return (
        summary_record.get("prediction_text")
        or summary_record.get("final_base_summary_text")
        or reward_record.get("prediction_text")
        or ""
    ).strip()

gpt_summary = by_tweet_id(paths["gpt_summary"])
t5_summary = by_tweet_id(paths["t5_summary"])
dpo_summary = by_tweet_id(paths["dpo_summary"])
gpt_reward = by_tweet_id(paths["gpt_reward"])
t5_reward = by_tweet_id(paths["t5_reward"])
dpo_reward = by_tweet_id(paths["dpo_reward"])

common_ids = sorted(set(gpt_summary) & set(t5_summary) & set(dpo_summary) & set(gpt_reward) & set(t5_reward) & set(dpo_reward))
print("common tweet IDs:", len(common_ids))

comparison_records = []
for tweet_id in common_ids:
    base = dpo_summary[tweet_id]
    gpt_r = gpt_reward[tweet_id]
    t5_r = t5_reward[tweet_id]
    dpo_r = dpo_reward[tweet_id]

    gpt_components = component_scores(gpt_r)
    t5_components = component_scores(t5_r)
    dpo_components = component_scores(dpo_r)

    record = {
        "tweet_id": tweet_id,
        "source_row_id": base.get("source_row_id"),
        "split": base.get("split"),
        "role": base.get("role"),
        "disaster_type": base.get("disaster_type"),
        "information_type": base.get("information_type"),
        "original_tweet": (
            gpt_summary[tweet_id].get("tweet_text")
            or ""
        ).strip(),
        "input_text": base.get("input_text", ""),
        "gpt_summary": candidate_text(gpt_summary[tweet_id], gpt_r),
        "t5_summary": candidate_text(t5_summary[tweet_id], t5_r),
        "dpo_summary": candidate_text(dpo_summary[tweet_id], dpo_r),
        "target_summary": base.get("target_text", ""),
        "gpt_reward": gpt_r["reward"],
        "t5_reward": t5_r["reward"],
        "dpo_reward": dpo_r["reward"],
        "dpo_minus_t5_reward": dpo_r["reward"] - t5_r["reward"],
        "dpo_minus_gpt_reward": dpo_r["reward"] - gpt_r["reward"],
        "gpt_components": gpt_components,
        "t5_components": t5_components,
        "dpo_components": dpo_components,
        "dpo_minus_t5_components": {
            key: dpo_components.get(key, 0.0) - t5_components.get(key, 0.0)
            for key in ["relevance", "factuality", "role_coverage", "urgency"]
        },
        "dpo_minus_gpt_components": {
            key: dpo_components.get(key, 0.0) - gpt_components.get(key, 0.0)
            for key in ["relevance", "factuality", "role_coverage", "urgency"]
        },
    }
    comparison_records.append(record)

records_by_role = defaultdict(list)
for record in comparison_records:
    records_by_role[record["role"]].append(record)

def select_unique(records, selectors):
    selected = []
    seen = set()
    for label, key_func, reverse in selectors:
        ordered = sorted(records, key=key_func, reverse=reverse)
        for candidate in ordered:
            if candidate["tweet_id"] not in seen:
                candidate = dict(candidate)
                candidate["sample_reason"] = label
                selected.append(candidate)
                seen.add(candidate["tweet_id"])
                break
    return selected

selectors = [
    ("largest_dpo_gain_vs_t5", lambda r: r["dpo_minus_t5_reward"], True),
    ("largest_dpo_loss_vs_t5", lambda r: r["dpo_minus_t5_reward"], False),
    ("largest_dpo_gain_vs_gpt", lambda r: r["dpo_minus_gpt_reward"], True),
    ("largest_dpo_loss_vs_gpt", lambda r: r["dpo_minus_gpt_reward"], False),
    ("largest_dpo_relevance_drop_vs_t5", lambda r: r["dpo_minus_t5_components"]["relevance"], False),
    ("median_dpo_reward", lambda r: abs(r["dpo_reward"] - sorted(x["dpo_reward"] for x in records_by_role[r["role"]])[len(records_by_role[r["role"]]) // 2]), False),
]

samples = []
for role in sorted(records_by_role):
    role_samples = select_unique(records_by_role[role], selectors)
    samples.extend(role_samples)

output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as out:
    for record in samples:
        out.write(json.dumps(record, ensure_ascii=False) + "\n")

print("wrote samples:", len(samples))
print("output:", output_path)

print("\nSample reasons by role:")
for role in sorted(records_by_role):
    role_sample_count = sum(1 for sample in samples if sample["role"] == role)
    print(role, role_sample_count)

print("\nFirst 3 JSONL samples:")
for record in samples[:3]:
    print(json.dumps(record, ensure_ascii=False)[:2000])
    print("---")

common tweet IDs: 6001
wrote samples: 42
output: /kaggle/working/role-aware-crisis-summarization/reports/tables/t5_dpo_v1_beta_0_1_qualitative_samples_by_role.jsonl

Sample reasons by role:
EMS 6
Firefighter 6
Firefighter/EMS 6
Police 6
Police/EMS 6
Police/Firefighter 6
Police/Firefighter/EMS 6

First 3 JSONL samples:
{"tweet_id": 5217, "source_row_id": 26044, "split": "train", "role": "EMS", "disaster_type": "Typhoon", "information_type": "Donations and volunteering", "original_tweet": "Support UNICEF’s emergency relief efforts for kids in the #Philippines. How to help:http://t.co/IjXIijeMNF #Haiyan http://t.co/GWLZU2SGws &amp;", "input_text": "Responder Roles: EMS\nDisaster Type: Typhoon\nTweet: Support UNICEF’s emergency relief efforts for kids in the #Philippines. How to help:http://t.co/IjXIijeMNF #Haiyan http://t.co/GWLZU2SGws &amp;\nSecondary Annotations: CERT\nInformation Type: Donations and volunteering", "gpt_summary": "EMS should evaluate medical needs and support for vulner

## Package T5-DPO Outputs

This cell packages the DPO checkpoint, predictions, metrics, reward scores, analysis reports, comparison tables, and qualitative inspection samples into one zip file for download and local inspection.

In [25]:
from pathlib import Path
import zipfile

repo = Path("/kaggle/working/role-aware-crisis-summarization")
zip_path = Path("/kaggle/working/t5_dpo_beta_0_1_outputs.zip")

files_and_dirs = [
    Path("/kaggle/working/t5_small_dpo_v1_beta_0_1"),
    repo / "data/modeling/t5_dpo_v1_beta_0_1",
    repo / "data/rewards/t5_dpo_v1_beta_0_1_reward_scores_tweet_relevance_minicheck.jsonl",
    repo / "reports/tables/t5_small_dpo_v1_beta_0_1_metrics.csv",
    repo / "reports/tables/t5_small_dpo_v1_beta_0_1_metrics.md",
    repo / "reports/tables/t5_dpo_v1_beta_0_1_reward_summary_tweet_relevance_minicheck.csv",
    repo / "reports/tables/t5_dpo_v1_beta_0_1_reward_summary_tweet_relevance_minicheck.md",
    repo / "reports/tables/t5_dpo_v1_beta_0_1_with_rewards_report.csv",
    repo / "reports/tables/summary_reward_analysis/t5_dpo_v1_beta_0_1_tweet_relevance_minicheck",
    repo / "reports/tables/gpt4o_vs_t5_v2_vs_t5_dpo_beta_0_1_reward_comparison.csv",
    repo / "reports/tables/gpt4o_vs_t5_v2_vs_t5_dpo_beta_0_1_reward_comparison_by_role.csv",
    repo / "reports/tables/t5_dpo_v1_beta_0_1_qualitative_samples_by_role.jsonl",
]

missing = [path for path in files_and_dirs if not path.exists()]
if missing:
    print("Missing expected artifacts:")
    for path in missing:
        print(path)
    raise FileNotFoundError("Some expected artifacts are missing.")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in files_and_dirs:
        if item.is_file():
            arcname = item.relative_to(repo) if item.is_relative_to(repo) else Path(item.name)
            zf.write(item, arcname)
        elif item.is_dir():
            for file in item.rglob("*"):
                if file.is_file():
                    if file.is_relative_to(repo):
                        arcname = file.relative_to(repo)
                    else:
                        arcname = Path(item.name) / file.relative_to(item)
                    zf.write(file, arcname)

print("Zip artifact:")
print(zip_path, f"{zip_path.stat().st_size / (1024 * 1024):.2f} MB")

print("\nPackaged artifacts:")
for item in files_and_dirs:
    if item.is_file():
        print("file", item, f"{item.stat().st_size / (1024 * 1024):.2f} MB")
    else:
        size = sum(file.stat().st_size for file in item.rglob("*") if file.is_file())
        print("dir ", item, f"{size / (1024 * 1024):.2f} MB")

Zip artifact:
/kaggle/working/t5_dpo_beta_0_1_outputs.zip 210.60 MB

Packaged artifacts:
dir  /kaggle/working/t5_small_dpo_v1_beta_0_1 232.85 MB
dir  /kaggle/working/role-aware-crisis-summarization/data/modeling/t5_dpo_v1_beta_0_1 4.39 MB
file /kaggle/working/role-aware-crisis-summarization/data/rewards/t5_dpo_v1_beta_0_1_reward_scores_tweet_relevance_minicheck.jsonl 10.89 MB
file /kaggle/working/role-aware-crisis-summarization/reports/tables/t5_small_dpo_v1_beta_0_1_metrics.csv 0.00 MB
file /kaggle/working/role-aware-crisis-summarization/reports/tables/t5_small_dpo_v1_beta_0_1_metrics.md 0.00 MB
file /kaggle/working/role-aware-crisis-summarization/reports/tables/t5_dpo_v1_beta_0_1_reward_summary_tweet_relevance_minicheck.csv 0.00 MB
file /kaggle/working/role-aware-crisis-summarization/reports/tables/t5_dpo_v1_beta_0_1_reward_summary_tweet_relevance_minicheck.md 0.00 MB
file /kaggle/working/role-aware-crisis-summarization/reports/tables/t5_dpo_v1_beta_0_1_with_rewards_report.csv 4.46 M